In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, hashlib, subprocess
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [2]:
# =============================================================================
# Cell 2 - THE LABEL-FREE MONITOR.
# At prediction time we do NOT know the true label. We know the predicted class
# (argmax) and the full calibrated score vector. So per PREDICTED class we watch
# the distribution of the PREDICTED-class APS score in the target stream and
# compare it to the SOURCE calibration distribution of that same predicted class.
# The drift statistic (KS between source-predicted and target-predicted score
# distributions) is fully label-free. We then ask: does it predict the TRUE
# per-class undercoverage measured in nb20?
#
# Deterministic mid-U APS score, matching nb20, for a reproducible distribution.
# =============================================================================
ALPHA = config.ALPHA_PRIMARY
def aps_all(P):
    order=np.argsort(-P,axis=1); sp=np.take_along_axis(P,order,1); cum=np.cumsum(sp,1)
    ss=cum-0.5*sp; out=np.empty_like(P); np.put_along_axis(out,order,ss,1); return out   # score per class
def pred_class(P): return np.argmax(P,axis=1)
MIN_SUPPORT=20   # below this many predicted-as-ci target flows, the monitor has no usable signal
def ks(a,b):
    if len(a)<5 or len(b)<5: return np.nan
    allv=np.sort(np.concatenate([a,b]))
    ca=np.searchsorted(np.sort(a),allv,side='right')/len(a)
    cb=np.searchsorted(np.sort(b),allv,side='right')/len(b)
    return float(np.max(np.abs(ca-cb)))
def qhat(s,a):
    n=len(s); return np.inf if n<1 else float(np.quantile(s,min(np.ceil((n+1)*(1-a))/n,1.0),method='higher'))

def monitor_rows(dataset, arch, classes, P_src, y_src, P_tgt, y_tgt):
    # scores for every class, source and target
    Ssrc=aps_all(P_src); Stgt=aps_all(P_tgt)
    yhat_src=pred_class(P_src); yhat_tgt=pred_class(P_tgt)
    out=[]
    for ci,cn in enumerate(classes):
        # ---- LABEL-FREE monitor: predicted-class score dist, source vs target ----
        s_src_pred = Ssrc[yhat_src==ci, ci]     # source flows PREDICTED ci -> their ci-score
        s_tgt_pred = Stgt[yhat_tgt==ci, ci]     # target flows PREDICTED ci -> their ci-score
        n_pred_tgt = int((yhat_tgt==ci).sum())
        n_pred_src = int((yhat_src==ci).sum())
        # If almost no target flows are predicted as ci, the monitor has NO signal for ci.
        # That is itself a detectable label-free failure mode (the class 'vanished' from
        # predictions), so we do NOT drop it: we flag it and mark the drift as unmeasurable.
        low_support = (n_pred_tgt < MIN_SUPPORT) or (n_pred_src < MIN_SUPPORT)
        drift_labelfree = np.nan if low_support else ks(s_src_pred, s_tgt_pred)
        # predicted-mass collapse: how much the predicted share of ci fell source->target
        share_src = n_pred_src/max(len(P_src),1); share_tgt = n_pred_tgt/max(len(P_tgt),1)
        pred_mass_drop = float(share_src - share_tgt)
        # ---- GROUND TRUTH (uses labels, only to evaluate the monitor) ----
        s_src_true = Ssrc[y_src==ci, ci]; s_tgt_true = Stgt[y_tgt==ci, ci]
        if len(s_src_true)<5 or len(s_tgt_true)<5: continue
        q=qhat(s_src_true,ALPHA); shc=float(np.mean(s_tgt_true<=q))
        under=(1-ALPHA)-shc
        drift_truelabel = ks(s_src_true, s_tgt_true)   # the oracle signal from nb20
        # misroute: fraction of TRUE-ci target flows predicted as something else.
        # high misroute = similarity-type (novel class read as familiar -> label-free blind)
        true_ci_tgt = (y_tgt==ci)
        misroute = float(np.mean(yhat_tgt[true_ci_tgt]!=ci)) if true_ci_tgt.any() else np.nan
        out.append({'dataset':dataset,'arch':arch,'class':cn,
                    'n_pred_target':n_pred_tgt,'low_support':bool(low_support),
                    'drift_labelfree':(np.nan if np.isnan(drift_labelfree) else round(drift_labelfree,4)),
                    'pred_mass_drop':round(pred_mass_drop,4),
                    'drift_truelabel':round(drift_truelabel,4),'misroute_target':round(misroute,4),
                    'SHC_coverage':round(shc,4),'undercoverage':round(under,4)})
    return out
print('label-free monitor ready; alpha =', ALPHA)


label-free monitor ready; alpha = 0.05


In [3]:
# =============================================================================
# Cell 3 - build monitor rows per architecture (seed-averaged within arch), CIC
# + UGR, same reconstruction as nb20.
# =============================================================================
ARCHS=['rf','xgb','mlp']
def avg_arch(prob_dir, key, arch):
    files=sorted(Path(prob_dir).glob(f'{key}{arch}__seed*.npz'))
    if not files: return None,None,0
    sp=tg=None; sh=None; k=0
    for f in files:
        d=np.load(f); s,t=d['srcpool'],d['target']
        if sh is None: sh=(s.shape,t.shape)
        elif (s.shape,t.shape)!=sh: raise ValueError(f'shape mismatch {f.name}')
        sp=s if sp is None else sp+s; tg=t if tg is None else tg+t; k+=1
    return sp/k, tg/k, k

rows=[]
cic=pd.read_parquet(config.INTERIM_DIR/'cicids2017_primary.parquet')
wed=cic[cic['day']=='wednesday'].reset_index(drop=True); wed=wed[wed['label'].isin(['DoS','Benign'])].reset_index(drop=True)
CICC=['Benign','DoS']; CIC_PROBS=config.DATA_DIR/'cic_probs'
REAL=['R1_holdout_Slowhttptest','R2_holdout_Slowloris','R3_holdout_GoldenEye',
      'R4_holdout_Slowloris_Slowhttptest','R5_holdout_GoldenEye_Slowloris']
def lab_cic(idx): return (wed.loc[idx,'label'].to_numpy()=='DoS').astype(int)
for name in REAL:
    spx=np.load(config.PROC_DIR/f'cic_{name}_srcpool_idx.npy'); tgx=np.load(config.PROC_DIR/f'cic_{name}_target_idx.npy')
    ysp,ytg=lab_cic(spx),lab_cic(tgx)
    for arch in ARCHS:
        Psp,Ptg,k=avg_arch(CIC_PROBS,f'{name}__',arch)
        if k: rows+=monitor_rows(f'cicids2017:{name}',arch,CICC,Psp,ysp,Ptg,ytg)

UGR=config.DATASETS_DIR/'ugr16'; usrc=pd.read_parquet(UGR/'july_week5.parquet'); utgt=pd.read_parquet(UGR/'august_week1.parquet')
for dd in (usrc,utgt): dd['label']=dd['label'].astype(str).str.strip().str.lower()
UK=['background','dos','scan11','scan44','nerisbotnet']
usrc=usrc[usrc.label.isin(UK)].reset_index(drop=True); utgt=utgt[utgt.label.isin(UK)].reset_index(drop=True)
UCL=sorted(UK); U2I={c:i for i,c in enumerate(UCL)}
def strat(df,fr,seed,col='label'):
    rng=np.random.default_rng(seed); nm=list(fr); f=np.array([fr[k] for k in nm],float); big=nm[int(np.argmax(f))]
    a=pd.Series(index=df.index,dtype=object)
    for _,s in df.groupby(col,sort=True):
        idx=s.index.to_numpy().copy(); rng.shuffle(idx); n=len(idx)
        c=np.floor(f*n).astype(int); c[nm.index(big)]+=n-c.sum(); kk=0
        for a2,q in zip(nm,c): a.loc[idx[kk:kk+q]]=a2; kk+=q
    return a
usrc=usrc.assign(partition=strat(usrc,config.SPLIT_FRACTIONS,20260725).values)
y_sp=usrc[usrc.partition=='source_cal_pool']['label'].map(U2I).to_numpy(); y_tg=utgt['label'].map(U2I).to_numpy()
for arch in ARCHS:
    Psp,Ptg,k=avg_arch(config.DATA_DIR/'ugr16_probs','ugr16__',arch)
    if k: rows+=monitor_rows('ugr16:july_to_august',arch,UCL,Psp,y_sp,Ptg,y_tg)

mon=pd.DataFrame(rows)
print('monitor rows:', len(mon))
print('\nUGR (label-free drift vs true undercoverage, mean over archs):')
um=mon[mon.dataset.str.startswith('ugr16')].groupby('class').agg(
   drift_labelfree=('drift_labelfree','mean'),drift_truelabel=('drift_truelabel','mean'),
   undercoverage=('undercoverage','mean'),n_pred_target=('n_pred_target','mean')).round(4).sort_values('undercoverage',ascending=False)
print(um.to_string())


monitor rows: 45

UGR (label-free drift vs true undercoverage, mean over archs):
             drift_labelfree  drift_truelabel  undercoverage  n_pred_target
class                                                                      
scan11                0.1362           0.5206         0.4903     35197.0000
scan44                0.2285           0.1993         0.1617     65148.6667
nerisbotnet           0.0763           0.0922        -0.0229     42129.0000
background            0.0455           0.0532        -0.0365    207463.6667
dos                   0.0080           0.0078        -0.0500     50061.6667


In [4]:
# =============================================================================
# Cell 4 - DOES THE LABEL-FREE MONITOR PREDICT UNDERCOVERAGE?
# Compare the label-free drift signal to the oracle (true-label) drift and to the
# actual undercoverage. If the label-free signal tracks undercoverage, the monitor
# works. We report both Spearman (ranking) and a threshold-based detection AUC
# (can it separate undercovering classes from safe ones).
# =============================================================================
from scipy import stats
from sklearn.metrics import roc_auc_score
m=mon.copy()
m['is_undercovering']=(m['undercoverage']>0.05).astype(int)   # >5pp below nominal = unsafe
n_low=int(m['low_support'].sum())
print(f'class-cells: {len(m)}  | low-support (monitor has no drift signal): {n_low}')

# COMBINED label-free detector: use the drift KS where measurable; where the class
# has collapsed out of the predictions (low support), fall back to predicted-mass
# drop, rescaled into a comparable [0,1]. This scores blind-spot classes instead of
# dropping them. Rank-based combine so scales are comparable.
m['lf_drift']=m['drift_labelfree']
r_drift=m['lf_drift'].rank(pct=True)
r_mass =m['pred_mass_drop'].clip(lower=0).rank(pct=True)
m['lf_detector']=np.where(m['low_support'], r_mass, r_drift)

print('\nSpearman vs undercoverage (ALL cells, blind spots kept):')
for sig,label in [('lf_detector','label-free detector (drift + mass-collapse)'),
                  ('drift_labelfree','drift only (measurable cells)'),
                  ('drift_truelabel','oracle (true-label drift)')]:
    sub=m.dropna(subset=[sig])
    rho,p=stats.spearmanr(sub[sig],sub['undercoverage']); print(f'  {label:42s} rho={rho:+.3f} p={p:.3g} (n={len(sub)})')

oo=m.dropna(subset=['drift_labelfree'])
rho_lo,p_lo=stats.spearmanr(oo['drift_labelfree'],oo['drift_truelabel'])
print(f'\nlabel-free vs oracle (measurable cells): rho={rho_lo:+.3f} p={p_lo:.3g} (n={len(oo)})')

print('\ndetection AUC (separate undercovering classes, undercoverage>5pp):')
det={}
if m['is_undercovering'].nunique()>1:
    det['lf_detector']=round(float(roc_auc_score(m['is_undercovering'], m['lf_detector'])),4)
    det['oracle']=round(float(roc_auc_score(m['is_undercovering'], m['drift_truelabel'])),4)
    print(f'  label-free detector  AUC={det["lf_detector"]:.3f}   (blind spots kept)')
    print(f'  oracle               AUC={det["oracle"]:.3f}')
print(f'  ({int(m.is_undercovering.sum())} undercovering / {len(m)} class-cells)')

print('\nlabel-free detector rho by architecture:')
for arch in sorted(m.arch.unique()):
    ma=m[m.arch==arch]; rho,_=stats.spearmanr(ma['lf_detector'],ma['undercoverage'])
    print(f'  [{arch}] rho={rho:+.3f} (n={len(ma)})')

rho_lf,p_lf=stats.spearmanr(m['lf_detector'],m['undercoverage'])
verdict={'monitor':'label-free per-class detector = predicted-class score-drift (KS) where measurable, '
            'else predicted-mass collapse; no target labels. Blind-spot classes scored, not dropped.',
   'spearman_detector_vs_undercoverage':{'rho':round(float(rho_lf),4),'p':float(p_lf)},
   'spearman_labelfree_vs_oracle_measurable':{'rho':round(float(rho_lo),4),'p':float(p_lo),'n':int(len(oo))},
   'detection_auc':det,'n_cells':int(len(m)),'n_low_support':int(n_low),
   'n_undercovering':int(m.is_undercovering.sum()),
   'limitations':['predicted-class drift for one class is contaminated by other classes misrouted into it',
                  'regime split uses a soft misroute axis, reported continuously not on a hard threshold'],
   'reads':('detector rho/AUC high => predicts coverage failure WITHOUT labels, including blind-spot '
            'classes via mass-collapse fallback. Boundary (cell 5) shows where drift alone goes blind.')}


class-cells: 45  | low-support (monitor has no drift signal): 0

Spearman vs undercoverage (ALL cells, blind spots kept):
  label-free detector (drift + mass-collapse) rho=+0.790 p=1.07e-10 (n=45)
  drift only (measurable cells)              rho=+0.790 p=1.07e-10 (n=45)
  oracle (true-label drift)                  rho=+0.895 p=1.17e-16 (n=45)

label-free vs oracle (measurable cells): rho=+0.841 p=4.64e-13 (n=45)

detection AUC (separate undercovering classes, undercoverage>5pp):
  label-free detector  AUC=0.942   (blind spots kept)
  oracle               AUC=0.964
  (20 undercovering / 45 class-cells)

label-free detector rho by architecture:
  [mlp] rho=+0.854 (n=15)
  [rf] rho=+0.816 (n=15)
  [xgb] rho=+0.782 (n=15)


In [ ]:
# =============================================================================
# Cell 5 - DISPLACEMENT vs SIMILARITY probe + figure + commit.
# The monitor should work where drift is DISPLACEMENT (model unsure -> predicted
# score visibly odd) and struggle where SIMILARITY (model confidently misreads a
# novel class as familiar -> predicted score looks normal). We proxy each class's
# regime by how often its TRUE members are predicted as SOMETHING ELSE in target
# (misroute rate): high misroute = similarity-type (label-free signal hidden in
# the WRONG predicted class). Recorded as the boundary evidence.
# =============================================================================
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

# ---- DISPLACEMENT vs SIMILARITY boundary: does misroute explain the blind spot? ----
from scipy import stats as _st
# keep blind-spot (low-support) classes: an unmeasurable drift signal = maximal
# blindness, not missing data, so drift_labelfree there is scored as 0 (saw nothing).
mm = mon.dropna(subset=['drift_truelabel','misroute_target']).copy()
mm['blindspot_gap'] = mm['drift_truelabel'] - mm['drift_labelfree'].fillna(0.0)
rho_b,p_b = _st.spearmanr(mm['misroute_target'], mm['blindspot_gap'])
print(f'boundary: Spearman(misroute, label-free blind-spot gap) rho={rho_b:+.3f} p={p_b:.3g}')
print('\nundercovering classes: is each DISPLACEMENT (low misroute, monitor catches) or SIMILARITY (high misroute, monitor blind)?')
uc = mm[mm['undercoverage']>0.05].groupby(['dataset','class']).agg(
     misroute=('misroute_target','mean'),drift_labelfree=('drift_labelfree','mean'),
     drift_truelabel=('drift_truelabel','mean'),undercoverage=('undercoverage','mean')).round(4)
uc['regime']=np.where(uc['misroute']>0.5,'similarity(high misroute)','displacement(low misroute)')
print(uc.to_string())
verdict['boundary_misroute_vs_blindspot']={'rho':round(float(rho_b),4),'p':float(p_b)}
verdict['undercovering_regimes']={f"{i[0]}|{i[1]}":uc.loc[i,'regime'] for i in uc.index}
(config.REPORTS_DIR/'monitor_verdict.json').write_text(json.dumps(verdict,indent=2))
mon.to_csv(config.REPORTS_DIR/'monitor_labelfree.csv', index=False)
(config.REPORTS_DIR/'monitor_verdict.json').write_text(json.dumps(verdict,indent=2))
print(json.dumps(verdict,indent=2))

fig,ax=plt.subplots(1,2,figsize=(11,4.4))
col={'rf':'tab:blue','xgb':'tab:green','mlp':'tab:red'}
for arch in sorted(mon.arch.unique()):
    s=mon[mon.arch==arch]
    ax[0].scatter(s['drift_labelfree'],s['undercoverage'],c=col.get(arch,'grey'),s=38,alpha=.75,label=arch)
    ax[1].scatter(s['drift_truelabel'],s['drift_labelfree'],c=col.get(arch,'grey'),s=38,alpha=.75,label=arch)
ax[0].axhline(0.05,color='grey',ls='--',lw=.8); ax[0].set_xlabel('label-free drift (KS, predicted-class score)')
ax[0].set_ylabel('true undercoverage'); ax[0].set_title('Monitor vs actual coverage failure'); ax[0].legend(title='arch',fontsize=8)
ax[1].plot([0,1],[0,1],color='grey',lw=.7); ax[1].set_xlabel('oracle drift (true-class score)')
ax[1].set_ylabel('label-free drift (predicted-class score)'); ax[1].set_title('Label-free vs oracle (gap = blind spot)')
fig.tight_layout(); fig.savefig(config.REPORTS_DIR/'monitor_labelfree.png',dpi=140); print('figure saved')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb21: label-free coverage-failure monitor (predicted-class score drift) + displacement/similarity probe')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


boundary: Spearman(misroute, label-free blind-spot gap) rho=+0.254 p=0.0929

undercovering classes: is each DISPLACEMENT (low misroute, monitor catches) or SIMILARITY (high misroute, monitor blind)?
                                                     misroute  drift_labelfree  drift_truelabel  undercoverage                      regime
dataset                                      class                                                                                        
cicids2017:R1_holdout_Slowhttptest           DoS       0.2461           0.3044           0.4086         0.2420  displacement(low misroute)
cicids2017:R2_holdout_Slowloris              DoS       0.4843           0.5218           0.4842         0.4343  displacement(low misroute)
cicids2017:R3_holdout_GoldenEye              DoS       0.2764           0.4115           0.3522         0.2278  displacement(low misroute)
cicids2017:R4_holdout_Slowloris_Slowhttptest DoS       0.8065           0.6155           0.8065         0.